In [2]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from astropy import units as u

In [5]:
def toa_calulator(df):
    '''
    Return the toa of a burst
    in the case of multiple components of a single burst, take the middle of the burst
    '''
    if df.shape[0] == 1:
        mjd = df.iloc[0]["toa_bary_tdb_inf_freq"]

    elif df.shape[0] > 1:
        mjd_list = df["toa_bary_tdb_inf_freq"].values
        mjd = (mjd_list[0] + mjd_list[-1]) / 2
 
    return mjd

In [6]:
def peak_sn_calc(df):
    '''
    input: pandas dataframe where each line has the burst properties per component
    Return the maximum value of the snr of all comp
    '''
    
    num_comp = df.shape[0]
    if num_comp == 1:
        peak_sn = df.iloc[0]["peak_snr"]
    elif num_comp > 1:
        peak_list = df["peak_snr"].values
        peak_sn = np.max(peak_list)
    
    return peak_sn,num_comp

In [7]:
def bandwidth_calc(df):
    '''
    input: pandas dataframe where each line has the burst properties per component
    return the bandwith of a burst
    '''
    
    obs_band = df.f0_MHz.values[0]
    
    #Pband
    if obs_band == 328.0:
        bw_obs = 54
        
    #Lband
    elif obs_band:
        bw_obs = 128
        
    #We take the absolute value, since the fwidth values can be negative
    #The bandwidth comes from autocorrelation and we define everything larger than FWHM as the BW
    signal_bw = int(abs(df.fwidth_sigma_MHz.values.max()) * 2.355 * 2)
    if signal_bw > bw_obs * 0.75:
        signal_bw = bw_obs
        
    return signal_bw, obs_band     

In [8]:
def isotropic_energies(fluences, distance=362.4):
    "Convert list of fluences to spectral densities"
    "Method based on fluence.py by K.Nimmo"

    #convert from jyms to jys
    fluence_jys = np.array(fluences) * 1e-3

    dis = 1.0 * u.megaparsec
    megap_cm = dis.to(u.cm).value

    #mpc to cm
    distance_lum_cm = megap_cm*distance

    #redshift correction, see evernote for details
    #redshift is from: Ravi et al, DSA-110
    #distance is also from Ravi et al, FAST converts themselves based on PLANCK 2016
    #link: https://ui.adsabs.harvard.edu/abs/2023ApJ...949L...3R/abstract
    z = 0.0771
    red_cor = 1 / (1 + z)**2

    energy_iso = fluence_jys * 4*np.pi*(distance_lum_cm**2) * 1e-23 * red_cor
    #print(energy_iso)

    return energy_iso

In [9]:
def isotropic_energies(fluences, distance=362.4):
    "Convert list of fluences to spectral densities"

    #convert from jyms to jys
    fluence_jys = np.array(fluences) * 1e-3

    dis = 1.0 * u.megaparsec
    megap_cm = dis.to(u.cm).value

    #mpc to cm
    distance_lum_cm = megap_cm*distance

    #redshift correction, see evernote for details
    #redshift is from: Ravi et al, DSA-110
    #distance is also from Ravi et al, FAST converts themselves based on PLANCK 2016
    #link: https://ui.adsabs.harvard.edu/abs/2023ApJ...949L...3R/abstract
    z = 0.0771
    red_cor = 1 / (1 + z)**2

    energy_iso = fluence_jys * 4*np.pi*(distance_lum_cm**2) * 1e-23 * red_cor
    #print(energy_iso)

    return energy_iso

In [10]:
def speclum(fluence, ontime, distance=362.4, z=0.0771):
    '''
    fluence assumed in Jy~ms
    ontime assumed in ms
    distance assumed in Mpc
    '''
    # actual ontime or total width? Width gives a lover limit on the luminosity.
    
    #convert Jy ms to J s; and ms to s
    fluence_jys = fluence*1e-3
    ontime /= 1e3
    
    #convert Mpc to cm
    distance_lum_cm = 3.086e24*distance
    energy_iso= fluence_jys*4*np.pi*(distance_lum_cm**2)*1e-23 / (1+z)**(2)
    lum_spec = energy_iso/ontime
    
    return lum_spec

In [23]:
def fluence_looper(df_fluence):
    """
    function to sum over the fluences components of bursts
    
    Input: The burst_csv file with all information, spc/sfxc/scale
    This is needed because we return a df with fluence (spc) and toa (sfxc)
    
    return: pandas dataframe with: id/fluence/telescope/c_freq/toa/width_ms
    """

    #the burst names to loop over
    exps = df_fluence['id'].unique()

    #Create a list with the telescope names for easy filtering later on
    exps_tel = [i.split('-')[1] for i in exps]

    #loop over every burst to sum the fluence
    info_list = []
    for exp, telescope in zip(exps, exps_tel):
        
        #Fluence: Making a small temp df for each exp, only taking the spc values
        if telescope != 'st':
            fluence_df_exp_temp = df_fluence[(df_fluence['id'] == exp) & (df_fluence['src'] == 'spc')]
            
            #TOA
            fluence_df_exp_temp_mjd = df_fluence[(df_fluence['id'] == exp) & (df_fluence['src'] == 'sfxc')]
            toa = toa_calulator(fluence_df_exp_temp_mjd) 
                
        elif telescope == 'st':
            fluence_df_exp_temp = df_fluence[df_fluence['id'] == exp]
            
            #TOA
            toa = toa_calulator(fluence_df_exp_temp) 
            
        #Calculate the total fluence
        fluence = sum(fluence_df_exp_temp['fluence_jyms'].values)
        
        #Calculate isotropic energies
        energies_isotropic = isotropic_energies(fluence)
        
        #Calculate the max SN of a burst
        peak_sn, num_comp = peak_sn_calc(fluence_df_exp_temp)

        #Calculate the bandwith
        signal_bw, cent_freq = bandwidth_calc(fluence_df_exp_temp)
        
        #width in ms
        width_t = fluence_df_exp_temp.end_acf_range.values.max() - fluence_df_exp_temp.begin_acf_range.values.min()

        #Calculate the spectral luminosity
        spel_lum = speclum(fluence, width_t, distance=362.4, z=0.0771)
        
        #Since the F0 is the same for all components, take all components, take the first entry
        cent_freq = fluence_df_exp_temp['f0_MHz'].values[0]
                            
        #Add to the list
        info_list.append([toa, peak_sn, fluence, num_comp, width_t, energies_isotropic, spel_lum, \
                          signal_bw, cent_freq])
        
    #Convert the fluence list to array
    info_list_arr = np.asarray(info_list)

    #Convert the info to dataframe
    new_data = {'id': exps, 'station': exps_tel, 'toa': info_list_arr[:,0], \
                'peak_sn': info_list_arr[:,1], 'fluence': info_list_arr[:,2], \
                'Number of components': info_list_arr[:,3],\
                'width_ms': info_list_arr[:,4], \
                'spectral density': info_list_arr[:,5],\
                'spectral luminosity': info_list_arr[:,6], \
                'bandwidth': info_list_arr[:,7], \
                'central frequency': info_list_arr[:,8]}
    
    df_new_fluence = pd.DataFrame(data=new_data)

    #add only the index number as a seperate column for potential later use
    #df_new_fluence['burst-index'] = df_new_fluence['id'].str.split('-').str[0].str[1:].astype(int)
    
    return df_new_fluence

In [24]:
def df_to_small_table(file):
    """
    Function to load in the burst.csv file and return the shortend version of the .csv file
    
    notes:
    B84  is missing, since it was determined to not be real after classification
    B130 is actively removed from calculations, since the RFI made it impossible to measure properties
    """

    #load in the main csv file
    fluence_df = pd.read_csv(file, index_col=0)
    fluence_df.sort_values(by=['id'])

    # Specifically remove B130-tr; because it was not possible to calculate a fluence
    mask = fluence_df['id'] == 'B130-tr'
    fluence_df = fluence_df[~mask]

    df_combined = fluence_looper(fluence_df)    
    df_combined.sort_values(by=['id'])
                          
    #Resort and reset the df
    #df_combined = df_combined.sort_values(by=['id'])
    #df_combined = df_combined.reset_index(drop=True)
    
    #Setting the formatting of the numbers correct
    df_combined["Number of components"] = df_combined["Number of components"].astype(int)
    df_combined["bandwidth"] = df_combined["bandwidth"].astype(int)
    df_combined["central frequency"] = df_combined["central frequency"].astype(int)
    
    df_combined['fluence_err'] = df_combined['fluence'] * 0.2
    df_combined['fluence_err'] = df_combined['fluence_err'].round(2)
    
    #Minimum spectral energy is on the order of 1e29 // calculated with the .min()
    df_combined['spectral density'] /= 1e29
    df_combined['spec_den_err'] = df_combined['spectral density'] * 0.2
    df_combined['pectral density'] = df_combined['spectral density'].round(2)
    df_combined['spec_den_err'] = df_combined['spec_den_err'].round(2)
    
    #Minimum spectral lum is on the order of 1e32 // calculated with the .min()
    df_combined['spectral luminosity'] /= 1e32
    df_combined['spec_lum_err'] = df_combined['spectral luminosity'] * 0.2
    df_combined['spectral luminosity'] = df_combined['spectral luminosity'].round(2)
    df_combined['spec_lum_err'] = df_combined['spec_lum_err'].round(2)
    
    df_combined['peak_sn'] = df_combined['peak_sn'].round(2)
    df_combined['width_ms'] = df_combined['width_ms'].round(2)
    df_combined['fluence'] = df_combined['fluence'].round(2)
    
    df_paper = df_combined[['id', 'station', 'toa', 'peak_sn', 'fluence', 'fluence_err', "Number of components",\
                            'width_ms', 'spectral density', 'spec_den_err', 'spectral luminosity', 'spec_lum_err',\
                            'bandwidth', 'central frequency']]

    name_csv = "FRB20220912A_table_paper.csv"
    df_paper.to_csv(name_csv, header=True, index=True, na_rep='NA')
    display(df_paper)

    return 

df_to_small_table("burst_stats_r117.csv")
#display(fluence_df)

,id,station,toa,peak_sn,fluence,fluence_err,Number of components,width_ms,spectral density,spec_den_err,spectral luminosity,spec_lum_err,bandwidth,central frequency
0,B01-st,st,59867.550262,14.90,42.16,8.43,2,9.61,57.106096,11.42,5.94,1.19,128,1381
1,B02-st,st,59868.720719,8.91,24.26,4.85,1,7.86,32.860919,6.57,4.18,0.84,128,1381
2,B03-st,st,59868.806998,4.69,29.73,5.95,1,13.54,40.273788,8.05,2.97,0.59,128,1381
3,B04-st,st,59868.952065,4.41,11.80,2.36,1,4.59,15.978530,3.20,3.48,0.70,128,1381
4,B05-tr,tr,59868.952070,16.67,40.75,8.15,1,3.97,55.194228,11.04,13.91,2.78,128,1414
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141,B127-wb,wb,59941.687390,6.99,274.25,54.85,1,20.48,371.465138,74.29,18.14,3.63,54,328
142,B128-wb,wb,59941.730176,24.17,2629.40,525.88,4,32.77,3561.491746,712.30,108.71,21.74,54,328
143,B129-st,st,59949.882698,6.78,46.15,9.23,1,12.67,62.510465,12.50,4.93,0.99,128,1381
144,B131-st,st,59978.532358,39.33,375.07,75.01,1,13.33,508.032688,101.61,38.13,7.63,128,1381


In [3]:
def max_values():
    
    # full_db = pd.read_csv('burst_stats_r117.csv', sep=",", header=0, index_col=0, na_values='NA', engine='python')
    # idx_max_jy = full_db['peak_flux_jy'].idxmax()
    # Filter_df = full_db[full_db.index.isin([idx_max_jy])]
    # display(Filter_df)

    table_db = pd.read_csv('FRB20220912A_table_paper.csv', sep=",", header=0, index_col=0, na_values='NA', engine='python')

    # idx_max_jy = table_db['fluence'].idxmax()
    # Filter_df = table_db[table_db.index.isin([idx_max_jy])]
    # display(Filter_df)

    # db_l = table_db[table_db['bandwidth'] != 54]
    # idx_max_jy = db_l['fluence'].idxmax()
    # Filter_df = db_l[db_l.index.isin([idx_max_jy])]
    # display(Filter_df)

    db_l = table_db[table_db['bandwidth'] != 54]
    widths = db_l["width_ms"].values
    print(np.median(widths))
    print(np.min(widths))
    print(np.max(widths))

    # db_l = table_db[table_db['bandwidth'] == 54]
    # widths = db_l["width_ms"].values
    # print(np.median(widths))
    # print(np.min(widths))
    
    return
max_values()

10.05
1.31
32.38
